In [53]:
from pathlib import Path
import pandas as pd

photo_folder = Path("C:/Users/omistaja/Desktop/AS Project/Data/Photos/S0001b")

files = list(photo_folder.glob("*"))

records = []

for file in files:
    records.append({
        "filename": file.name,
        "path": str(file)
    })

import re
import pandas as pd

manifest = pd.DataFrame(records)

manifest.head(445)

,filename,path
0,SG0031_O0031_P0054.JPG,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
1,SG0032_O0032_P0055.JPG,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
2,SG0033_O0033_P0056.JPG,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
3,SG0034_O0034_P0057.JPG,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
4,SG0035_O0035_P0058.JPG,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
5,SG0036_O0036_P0059.JPG,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
6,SG0037_O0037_P0060.JPG,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
7,SG0038_O0038_P0061.JPG,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
8,SG0039_O0039_P0062.JPG,C:\Users\omistaja\Desktop\AS Project\Data\Phot...
9,SG0041_O0041_P0064.JPG,C:\Users\omistaja\Desktop\AS Project\Data\Phot...


In [54]:
import pandas as pd

manifest = pd.DataFrame({
    "path": files
})

manifest["filename"] = manifest["path"].apply(
    lambda x: x.name
)

manifest.head(445)

manifest.to_csv("Filenames")

In [55]:
test_file = manifest.iloc[0]["path"]

print(test_file)

C:\Users\omistaja\Desktop\AS Project\Data\Photos\S0001b\SG0031_O0031_P0054.JPG


In [56]:
from PIL import Image
from PIL.ExifTags import TAGS

with Image.open(test_file) as img:
    exif = img.getexif()

    print("Number of EXIF fields:", len(exif))

    for tag_id, value in exif.items():
        tag = TAGS.get(tag_id, tag_id)
        print(tag, ":", value)

Number of EXIF fields: 12
GPSInfo : 9300
ResolutionUnit : 2
ExifOffset : 360
Make : Canon
Model : Canon EOS 100D
YResolution : 72.0
Orientation : 8
DateTime : 2026:08:16 11:01:26
YCbCrPositioning : 2
Copyright : 
XResolution : 72.0
Artist : 


In [57]:
from PIL import Image
from PIL.ExifTags import IFD
import pandas as pd

def get_timestamp(file_path):
    try:
        with Image.open(file_path) as img:
            exif = img.getexif()

            exif_ifd = exif.get_ifd(IFD.Exif)

            timestamp = exif_ifd.get(36867)

            return timestamp

    except Exception:
        return None

In [58]:
manifest["capture_datetime_original"] = (
    manifest["path"].apply(get_timestamp)
)

manifest

,path,filename,capture_datetime_original
0,C:\Users\omistaja\Desktop\AS Project\Data\Phot...,SG0031_O0031_P0054.JPG,2026:08:16 11:01:26
1,C:\Users\omistaja\Desktop\AS Project\Data\Phot...,SG0032_O0032_P0055.JPG,2026:08:16 11:04:07
2,C:\Users\omistaja\Desktop\AS Project\Data\Phot...,SG0033_O0033_P0056.JPG,2026:08:16 11:05:20
3,C:\Users\omistaja\Desktop\AS Project\Data\Phot...,SG0034_O0034_P0057.JPG,2026:08:16 11:06:51
4,C:\Users\omistaja\Desktop\AS Project\Data\Phot...,SG0035_O0035_P0058.JPG,2026:08:16 11:07:09
5,C:\Users\omistaja\Desktop\AS Project\Data\Phot...,SG0036_O0036_P0059.JPG,2026:08:16 11:07:26
6,C:\Users\omistaja\Desktop\AS Project\Data\Phot...,SG0037_O0037_P0060.JPG,2026:08:16 11:09:23
7,C:\Users\omistaja\Desktop\AS Project\Data\Phot...,SG0038_O0038_P0061.JPG,2026:08:16 11:09:35
8,C:\Users\omistaja\Desktop\AS Project\Data\Phot...,SG0039_O0039_P0062.JPG,2026:08:16 11:11:22
9,C:\Users\omistaja\Desktop\AS Project\Data\Phot...,SG0041_O0041_P0064.JPG,2026:08:16 11:12:07


In [59]:
manifest["capture_datetime_original"] = pd.to_datetime(
    manifest["capture_datetime_original"],
    format="%Y:%m:%d %H:%M:%S",
    errors="coerce"
)

manifest[
    ["filename", "capture_datetime_original"]
].head(10)

camera_offset_hours = 5

manifest["capture_datetime_corrected"] = (
    manifest["capture_datetime_original"]
    + pd.Timedelta(hours=camera_offset_hours)
)

manifest[
    [
        "filename",
        "capture_datetime_original",
        "capture_datetime_corrected"
    ]
].head(1500)

manifest.to_csv("S0002_photo_timestamps_corrected")

In [8]:
%pip install gpxpy

Note: you may need to restart the kernel to use updated packages.


In [60]:
from pathlib import Path
import gpxpy
import pandas as pd

gpx_path = Path(
    "C:/Users/omistaja/Desktop/AS Project/Data/Comaps/Xinshi_Township_08162026_S0001b.gpx"
)

with open(gpx_path, "r", encoding="utf-8") as gpx_file:
    gpx = gpxpy.parse(gpx_file)

track_points = []

for track in gpx.tracks:
    for segment in track.segments:
        for point in segment.points:
            track_points.append({
                "gpx_time": point.time,
                "latitude": point.latitude,
                "longitude": point.longitude,
                "elevation": point.elevation
            })

gpx_df = pd.DataFrame(track_points)

gpx_df["gpx_time_taiwan"] = (
    pd.to_datetime(gpx_df["gpx_time"], utc=True)
    .dt.tz_convert("Asia/Taipei")
)

gpx_df["gpx_time_local"] = (
    gpx_df["gpx_time_taiwan"]
    .dt.tz_localize(None)
)

gpx_df.head(600)

,gpx_time,latitude,longitude,elevation,gpx_time_taiwan,gpx_time_local
0,2026-08-16 08:03:10+00:00,23.103176,120.28576,9.0,2026-08-16 16:03:10+08:00,2026-08-16 16:03:10
1,2026-08-16 08:03:21+00:00,23.103178,120.28587,11.0,2026-08-16 16:03:21+08:00,2026-08-16 16:03:21
2,2026-08-16 08:03:29+00:00,23.103202,120.28597,11.0,2026-08-16 16:03:29+08:00,2026-08-16 16:03:29
3,2026-08-16 08:03:41+00:00,23.103223,120.28609,11.0,2026-08-16 16:03:41+08:00,2026-08-16 16:03:41
4,2026-08-16 08:03:57+00:00,23.103243,120.28619,11.0,2026-08-16 16:03:57+08:00,2026-08-16 16:03:57
5,2026-08-16 08:04:07+00:00,23.103260,120.28631,11.0,2026-08-16 16:04:07+08:00,2026-08-16 16:04:07
6,2026-08-16 08:04:16+00:00,23.103285,120.28642,10.0,2026-08-16 16:04:16+08:00,2026-08-16 16:04:16
7,2026-08-16 08:07:11+00:00,23.103368,120.28649,11.0,2026-08-16 16:07:11+08:00,2026-08-16 16:07:11
8,2026-08-16 08:08:53+00:00,23.103337,120.28659,11.0,2026-08-16 16:08:53+08:00,2026-08-16 16:08:53
9,2026-08-16 08:09:01+00:00,23.103349,120.28670,11.0,2026-08-16 16:09:01+08:00,2026-08-16 16:09:01


In [61]:
manifest[
    ["filename", "capture_datetime_corrected"]
].head(20)

,filename,capture_datetime_corrected
0,SG0031_O0031_P0054.JPG,2026-08-16 16:01:26
1,SG0032_O0032_P0055.JPG,2026-08-16 16:04:07
2,SG0033_O0033_P0056.JPG,2026-08-16 16:05:20
3,SG0034_O0034_P0057.JPG,2026-08-16 16:06:51
4,SG0035_O0035_P0058.JPG,2026-08-16 16:07:09
5,SG0036_O0036_P0059.JPG,2026-08-16 16:07:26
6,SG0037_O0037_P0060.JPG,2026-08-16 16:09:23
7,SG0038_O0038_P0061.JPG,2026-08-16 16:09:35
8,SG0039_O0039_P0062.JPG,2026-08-16 16:11:22
9,SG0041_O0041_P0064.JPG,2026-08-16 16:12:07


In [62]:
gpx_df[
    ["gpx_time_taiwan", "latitude", "longitude"]
].head(445)

,gpx_time_taiwan,latitude,longitude
0,2026-08-16 16:03:10+08:00,23.103176,120.28576
1,2026-08-16 16:03:21+08:00,23.103178,120.28587
2,2026-08-16 16:03:29+08:00,23.103202,120.28597
3,2026-08-16 16:03:41+08:00,23.103223,120.28609
4,2026-08-16 16:03:57+08:00,23.103243,120.28619
5,2026-08-16 16:04:07+08:00,23.103260,120.28631
6,2026-08-16 16:04:16+08:00,23.103285,120.28642
7,2026-08-16 16:07:11+08:00,23.103368,120.28649
8,2026-08-16 16:08:53+08:00,23.103337,120.28659
9,2026-08-16 16:09:01+08:00,23.103349,120.28670


In [63]:
manifest = manifest.sort_values(
    "capture_datetime_corrected"
).reset_index(drop=True)

gpx_df = gpx_df.sort_values(
    "gpx_time_local"
).reset_index(drop=True)

matched = pd.merge_asof(
    manifest,
    gpx_df[
        ["gpx_time_local", "latitude", "longitude"]
    ],
    left_on="capture_datetime_corrected",
    right_on="gpx_time_local",
    direction="nearest"
)

matched.head(445)

matched.to_csv("Photo coordinates2")

In [13]:
matched["time_difference_seconds"] = (
    matched["capture_datetime_corrected"]
    - matched["gpx_time_local"]
).abs().dt.total_seconds()

matched[
    [
        "filename",
        "capture_datetime_corrected",
        "gpx_time_local",
        "time_difference_seconds",
        "latitude",
        "longitude"
    ]
].head(20)

matched["time_difference_seconds"].describe()

matched["gpx_match_status"] = matched["time_difference_seconds"].apply(
    lambda x: "good" if x <= 30 else "review"
)

matched["gpx_match_status"].value_counts()

gpx_match_status
good      323
review    122
Name: count, dtype: int64

In [14]:
matched["time_difference_seconds"].describe()

pd.cut(
    matched["time_difference_seconds"],
    bins=[0, 5, 10, 30, 60, 120, 300, float("inf")],
    include_lowest=True
).value_counts().sort_index()

time_difference_seconds
(-0.001, 5.0]     109
(5.0, 10.0]        77
(10.0, 30.0]      137
(30.0, 60.0]       74
(60.0, 120.0]      41
(120.0, 300.0]      5
(300.0, inf]        2
Name: count, dtype: int64

In [15]:
matched[
    [
        "filename",
        "capture_datetime_corrected",
        "gpx_time_local",
        "time_difference_seconds",
        "latitude",
        "longitude"
    ]
].sort_values(
    "time_difference_seconds",
    ascending=False
).head(30)

,filename,capture_datetime_corrected,gpx_time_local,time_difference_seconds,latitude,longitude
0,IMG_0659.JPG,2026-08-18 12:56:10,2026-08-18 13:02:47,397.0,24.847607,121.55125
1,IMG_0660.JPG,2026-08-18 12:56:22,2026-08-18 13:02:47,385.0,24.847607,121.55125
2,IMG_0665.JPG,2026-08-18 12:58:50,2026-08-18 13:02:47,237.0,24.847607,121.55125
3,IMG_0666.JPG,2026-08-18 12:59:01,2026-08-18 13:02:47,226.0,24.847607,121.55125
4,IMG_0668.JPG,2026-08-18 13:00:03,2026-08-18 13:02:47,164.0,24.847607,121.55125
5,IMG_0669.JPG,2026-08-18 13:00:04,2026-08-18 13:02:47,163.0,24.847607,121.55125
134,IMG_0825.JPG,2026-08-18 13:39:51,2026-08-18 13:37:45,126.0,24.848721,121.55170
133,IMG_0824.JPG,2026-08-18 13:39:45,2026-08-18 13:37:45,120.0,24.848721,121.55170
135,IMG_0826.JPG,2026-08-18 13:41:09,2026-08-18 13:43:09,120.0,24.848801,121.55180
27,IMG_0706.JPG,2026-08-18 13:14:41,2026-08-18 13:12:46,115.0,24.847857,121.55138


In [123]:
target_time = pd.Timestamp("2026-08-18 07:58:50")

closest = gpx_df.iloc[
    (gpx_df["gpx_time_local"] - target_time)
    .abs()
    .argsort()[:1]
]

closest[
    ["gpx_time_local", "latitude", "longitude"]
]

,gpx_time_local,latitude,longitude
0,2026-08-18 13:02:47,24.847607,121.55125
